# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/40/f4b_k5mj6hj9b144z92x4cb40000gn/T/ipykernel_54558/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [4]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [5]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [6]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [14]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata is important in RAG because it provides structured information about documents and chunks that can be used for filtering, security, freshness, ranking and traceability. It helps ensure the retrieval system returns the most relevant and authorised documents rather than relying solely on semantic similarity. For example, location, department, doc type, creation date, author, permissions and source of information.

Better RAG =! better embedding

Better RAG = better metadata strategy

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [15]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [16]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:
Choosing chunk size and chunk overlap is a tradeoff between retrieval precision vs context preservation. These affects storage costs and retrieval quality. Larger chunk constain more context but may retrieve irrelevant information with lowered retrieval precision. For example, chunk = 1000 tokens contain topic of Employee leave policy, Eligibility rules, Exceptions
Approval process. The user query asks "what is parental leave eligibility?". Although big chunk contains rich context, it introduces noises as only 50 tokens are relevant to the query.

Smaller Chunk does the opposite. It improves retrieval precision but can lose important context. For example, chunk = 100 tokens contain the topic of Eligibility rules which is very precise but it may spilt eligibility in 1 chunk and exception in the next chunk. The model loses the full picture and will return incomplete information.

This can be fixed with Chunk overlap as it helps preserve information that crosses chunk boundaries, but increases storage requirements and retrieval redundancy.

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [17]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [18]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [22]:
retrieval_k = 7
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

In [29]:
retrieval_query="How do I bake sourdough bread?"
display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.106 | page=14 | start_index=4012
to discuss the importance of oral health at kitten wellness appoint- ments, the owner will come to think of the cat’s dental health as being a signiﬁcant contributor to its quality of life. 107 After the practitioner has determined that no malocclusion or dental eruption problems are present, 108 practice team members can instruct owners on how to 
--------------------------------------------------------------------------------
Source 2 | score=0.101 | page=9 | start_index=4756
istration of medications both orally and subcutaneously. Ultimately, almost every cat is going to require medication at some time in its life, so it is prudent to acclimate cats to these types of procedures. Kittens may be taught to accept pilling by administration of a tasty morsel of food instead of a pill. By giving treats that are soft enough t
--------------------------------------------------------------------------------
Source 3 | score=0.096 | page=9 |

[(Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 13, 'page_label': '14', 'document_type': 'cat_health_guideline', 'start_index': 4012, '_id': '215f658a54014800a793be90c9a59945', '_collection_name': 'cat_health_guidelines'}, page_content='to discuss the importance of oral health at kitten wellness appoint-\nments, the owner will come to think of the cat’s dental health as being a\nsigniﬁcant contributor to its quality of life.\n107 After the practitioner has\ndetermined that no malocclusion or dental eruption problems are\npresent,\n108 practice team members can instruct owners on how to ex-\namine the cat’s mouth and how to brush the teeth. Providing videos,\nwritten and verbal instructions, and samples of products tha

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

Similarity scores help me understands how far apart of the query and the retrieval are. Further away is less relevant and closer together is more relevant. However, the score doesn't prove whether the chunk actually answers the questions or whether the answer is medically correct or safe. This score is not an indication of absolute confidence - 0.58 isn't 58% sure - it's just for the relative ranking .

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [30]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [31]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [32]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [33]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that your cat may need veterinary attention can include:

- **Pain or anxiety/stress signs**, especially if they’re new or persistent [Source 1][Source 4]
- **Changes in grooming** — increased grooming can point to a skin or other medical issue, and **reduced grooming** may indicate illness, bladder pain, joint pain, or reduced mobility [Source 1][Source 2]
- **Vomiting, vomiting hairballs, or diarrhea**, especially if frequent or accompanied by other changes [Source 3]
- **Changes in appetite** [Source 3]
- **Increased thirst and urination** [Source 3]
- **Increased nocturnal activity, vocalization, or changes in normal habits/activity** [Source 3]
- **Behavior changes** such as house-soiling or aggression [Source 2]
- **Stress/fear body language**, such as crouching, hiding, flattening ears, dilated pupils, tense posture, frantic fleeing, hissing, yowling, growling, or screaming [Source 4]

If you’re seeing any of these signs, especially if they’re ongoing or worsening, contact

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [34]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guideline recommends **individualized preventive care** based on a cat’s life stage and risk factors, with at least **annual veterinary examinations for all cats** and **every 6 months for senior cats** or more often if they have chronic conditions [Source 4]. It also notes that **routine, regular use of broad-spectrum parasite prevention** is likely beneficial for most pet cats, and that cats with outdoor exposure, travel, boarding, or grooming visits may need extra parasite control measures [Source 1].
Question: What symptoms should make me call a veterinarian?
You should call a veterinarian if your cat has changes in appetite, increased urination or thirst, vomiting, vomiting hairballs, diarrhea, or weight changes. Also contact a vet if you notice increased nocturnal activity or vocalization, or changes in normal habits, activity, demeanor, jumping, or climbing, since these can be signs of disease, pain, mobility issues, or

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

For the first 3 queries, retrieved context looked relevant because there are citations to the source.

The query about taxes, the retrieved context was not relevant - vector shear still return cats health chunks but it doesn't support the question. The generation rule inside of system prompt caught this irrelevancy with "I don't have enough information in the provided cat health guideline PDF to answer that."

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

### Pick on test question and inspect the retrieved chunks and scores

In [39]:
answer_k = 5

result = answer_question(
    "What are commons illness in kitten?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Common kitten health issues mentioned in the provided guideline include:
- Upper respiratory or parasitic disease signs, especially depending on lifestyle and exposure [Source 2]
- Congenital issues such as a heart murmur, hernia, or cleft palate [Source 1]
- Dentition abnormalities seen on oral exam [Source 1]
- Behavior concerns in orphaned or undersocialized kittens [Source 2]
- Lower airway disease is also noted as common in young adult cats, not specifically kittens [Source 1]

I don’t have enough information in the provided cat health guideline PDF to give a fuller list of common kitten illnesses.

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 7, 'start_index': 820, 'score': 0.5510975617869172}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 7, 'start_index': 0, 'score': 0.4942247330287372}
{'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1549, 'score': 0.48697674570104155}


In [41]:
#checking retrieval score

retrieval_k_quality = 5

retrieval_query_quality="What are commons illness in kitten?"
display_retrieval_results(retrieval_query_quality, k=retrieval_k_quality)


Source 1 | score=0.551 | page=7 | start_index=820
24 Changes in demeanor, activity level, and behavior are additionally key to note and trend over time. Asking speci ﬁc questions as to whether the kitten is displaying any unwanted behaviors, counselling clients on normal kitten be- havior, and giving advice on positive methods to modify unwanted behavior are critical discussion points at this stag
--------------------------------------------------------------------------------
Source 2 | score=0.494 | page=7 | start_index=0
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require additional focus during the examination by each life stage are listed in Table 3. Kittens
--------------------------------------------------------------------------------
Source 3 | score=0.487 | page=15 | sta

[(Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 6, 'page_label': '7', 'document_type': 'cat_health_guideline', 'start_index': 820, '_id': 'bc3bc951f9234861aca24b786215e7f9', '_collection_name': 'cat_health_guidelines'}, page_content='24 Changes in\ndemeanor, activity level, and behavior are additionally key to note\nand trend over time.\nAsking speci ﬁc questions as to whether the kitten is displaying\nany unwanted behaviors, counselling clients on normal kitten be-\nhavior, and giving advice on positive methods to modify unwanted\nbehavior are critical discussion points at this stage. Breed-related\npredispositions, signs of genetic disease, and the availability and\naccuracy of genetic testing to detect disease shou

### Rebuild the splitter and vector store.

In [102]:
chunk_size_quality = 500
chunk_overlap_quality  = 100

text_splitter_quality = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size_quality,
    chunk_overlap=chunk_overlap_quality,
    add_start_index=True,
)

splits_quality = text_splitter_quality.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits_quality)} chunks.")
print(f"Chunk size: {chunk_size_quality} characters")
print(f"Chunk overlap: {chunk_overlap_quality} characters")

Split 22 pages into 263 chunks.
Chunk size: 500 characters
Chunk overlap: 100 characters


In [103]:
collection_name_quality = "cat_health_guidelines_quality"

vector_store_quality= QdrantVectorStore.from_documents(
    documents=splits_quality,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name_quality,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name_quality}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines_quality


In [104]:
def display_retrieval_results_quality(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results_quality = vector_store_quality.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results_quality, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results_quality

### Rerun the retrieval query with the new vector store

In [105]:
display_retrieval_results_quality(retrieval_query_quality, k=retrieval_k_quality)

Source 1 | score=0.513 | page=12 | start_index=1954
is noted by the owner, the kitten should be evaluated for underlying conditions such as congenital abnormalities of the lower urinary or GI tract, GI parasites, or other infectious diseases. Mature adult and senior cats may house-soil secondarily to medical or behavioral conditions. Clients should be encouraged to seek veterinary assistance promptl
--------------------------------------------------------------------------------
Source 2 | score=0.495 | page=7 | start_index=421
and history, including exposure to other cats and the level of care provided. V accination and parasite control history, health status of related cats, if known, and clinical signs of upper respiratory or parasitic disease are all important areas of focus. Nutritional status and weaning history are also important areas of inquiry as orphaned or und
--------------------------------------------------------------------------------
Source 3 | score=0.487 | page=7 | 

[(Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 11, 'page_label': '12', 'document_type': 'cat_health_guideline', 'start_index': 1954, '_id': '61db0796974a46c7929c1b416b4f6e63', '_collection_name': 'cat_health_guidelines_quality'}, page_content='is noted by the owner, the kitten should be evaluated for underlying\nconditions such as congenital abnormalities of the lower urinary or GI\ntract, GI parasites, or other infectious diseases. Mature adult and senior\ncats may house-soil secondarily to medical or behavioral conditions.\nClients should be encouraged to seek veterinary assistance promptly, in\norder to diagnose life-threatening conditions such as urinary tract\nblockage, and to avoid having the behavior become en

In [110]:
def answer_question_quality(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs_quality = vector_store_quality.similarity_search_with_score(question, k=k)
    context_quality = format_context(scored_docs_quality)
    answer_quality = rag_chain.invoke({"context": context_quality, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs_quality, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer_quality,
        "sources": sources,
        "context": scored_docs_quality,
    }

In [114]:
answer_k_quality = 7

result_quality = answer_question_quality(
    "What are commons illness in kitten?",
    k=answer_k_quality,
)

print(result_quality["answer"])
print("\nSources:")
for source in result_quality["sources"]:
    print(source)

I don’t have enough information in the provided cat health guideline PDF to answer that. The closest relevant details mention that kittens should be evaluated for underlying conditions such as congenital abnormalities of the lower urinary or GI tract, GI parasites, or other infectious diseases, and that kitten history should include vaccine/parasite control, respiratory signs, nutrition, and weaning history [Source 1][Source 2]. For any illness concern, please contact a veterinarian.

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 12, 'start_index': 1954, 'score': 0.5126042006587412}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 7, 'start_index': 421, 'score': 0.4949147420147882}
{'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 7, 'start_index': 2785, 'score': 0.4868381525514139}
{'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1579, 'score': 0.4760480430

In [115]:

answer_k_quality = 7

result_quality = answer_question_quality(
    "What diseases should veterinarians screen for in kittens?",
    k=answer_k_quality,
)

print(result_quality["answer"])
print("\nSources:")
for source in result_quality["sources"]:
    print(source)

In kittens, the guideline highlights screening focus on upper respiratory disease and parasitic disease, along with related history such as vaccination and parasite control. It also notes attention to infectious diseases and GI parasites when certain signs are present. [Source 1] [Source 6]

The vaccine-related diseases specifically mentioned for kittens include feline panleukopenia virus (FPV), feline herpesvirus-1 (FHV-1), feline calicivirus (FCV), and feline leukemia virus (FeLV). [Source 2] [Source 4]

If you mean what to actively test for based on risk or signs, I’d recommend asking a veterinarian, since screening depends on lifestyle and exposure risk.

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 7, 'start_index': 421, 'score': 0.5723550235660589}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 4028, 'score': 0.5701269031335463}
{'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'pa

### 🏗️ Activity Notes


**Settings tried:** `chunk_size` 1000 → 500, `chunk_overlap` 200 → 100; `k` (5 vs 7); query wording

**Test questions:**
  1. Broad: common illnesses / health issues in kittens
  2. Narrower (on tuned vector store): what diseases veterinarians should screen for in kittens

**Broad question — baseline (`1000/200`) vs tuned (`500/100`):**
- Before: top score 0.551; sources from pages 7, 15, 19, 21; answer covered URI/parasites, congenital issues, dentition, and behavior
- After: top score 0.513; top chunk from page 12 (house-soiling workup); answer narrower (GI/urinary congenital issues, parasites, URI signs)

**Result:** Retrieval did not improve for this broad question. Baseline returned broader, more useful context.

**Changing `k` only:**
- Increased how many chunks were sent to the LLM, but did not change ranking or match quality.

**Query wording on tuned store (`500/100`):**
- Rephrased to a screening-focused question (veterinary screening / URI, parasites, FeLV, history)
- Top scores rose to ~0.55-0.57 (top: 0.572); sources on pages 7 and 15 aligned with kitten screening content
- Answer was more focused and better matched how the PDF is written, while still noting the PDF has no full disease list

**Did retrieval improve overall?**
- Not for smaller chunks alone on a broad question
- Yes when query wording matched document language, even on the tuned index

**Takeaway:** Chunk size and query wording work together. Smaller chunks helped for a specific, document-aligned question but hurt for a broad overview question. Similarity scores are not directly comparable across different chunk indexes; judge retrieval by source relevance, chunk previews, and answer completeness—not the top score alone.